# Oxta — Stack Smoke Test (200M params, full stack)

**O que esse notebook valida:** que o stack COMPLETO do Oxta — Mamba2 + Attention + MoE-8 + BitNet 1.58 QAT + BF16 Tensor Cores + AVX2/CUDA — funciona junto e o modelo **aprende** (loss desce) num modelo de ~200M params.

**O que NÃO valida:** qualidade do modelo final, generalização, performance em tarefas reais. É apenas "o stack tá vivo?".

**Critério único de sucesso:** loss inicial em ~10-12 (ln(vocab) inicial) cai pra <8 nos primeiros 500 steps.

**Tempo estimado:** 35-50 min em T4 (build + dataset pequeno + 500 steps).

## Stack que vamos exercitar

| Componente | Configuração | Por quê |
|-----------|--------------|---------|
| Mamba2 SSD | 7 de 12 layers | base, state space duality |
| Attention (GQA) | 3 de 12 layers, period=4 | razão Mamba/Attn 7:3 da v11 |
| MoE-8 (top-2) | 2 de 12 layers, period=6 | sparse experts |
| BitNet 1.58 (QAT) | warmup 80 → start 160 | ternarização progressiva |
| BF16 mixed precision | `NSOS_MIXED_PRECISION=bf16` | Tensor Cores T4 sm_75 |
| Gradient checkpointing | on | T4 16GB precisa |
| CUDA + AVX2 | on | always |

Target: ~180-220M params (d_model=768, num_layers=12, vocab=16384, MoE-8 em 2 layers).

## 1. Setup Colab (Drive + repo + build)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_OWNER = 'vitorGgC569'
REPO_NAME = 'reimagined'
BRANCH = 'oxta-contabil'
REPO_ROOT = Path('/content/reimagined-main')

# ── Auth: tenta GITHUB_TOKEN do Colab Secrets (necessário se repo é privado)
# Setup: clica no ícone 🔑 na sidebar esquerda do Colab → Add new secret
#        Name: GITHUB_TOKEN  | Value: ghp_... (fine-grained PAT com Contents:read)
#        Toggle "Notebook access" ON
def build_clone_url():
    public_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        if token:
            print('[auth] usando GITHUB_TOKEN do Colab Secrets')
            return f'https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
    except Exception:
        pass
    print('[auth] sem GITHUB_TOKEN — tentando clone público')
    return public_url

clone_url = build_clone_url()

def explain_auth_failure():
    print('=' * 70)
    print('CLONE FALHOU — provavelmente o repo é PRIVADO e sem auth.')
    print('=' * 70)
    print('')
    print('CAMINHO A — tornar repo público (mais rápido):')
    print(f'  github.com/{REPO_OWNER}/{REPO_NAME}/settings')
    print('  -> Change repository visibility -> Public')
    print('')
    print('CAMINHO B — Personal Access Token (mais seguro):')
    print('  1. github.com/settings/personal-access-tokens/new')
    print(f'     - Repository access: Only "{REPO_OWNER}/{REPO_NAME}"')
    print('     - Permissions > Contents: Read-only')
    print('  2. No Colab, sidebar esquerda → ícone 🔑 "Secrets"')
    print('  3. + Add new secret:')
    print('       Name:  GITHUB_TOKEN')
    print('       Value: <cola o token ghp_... aqui>')
    print('  4. Toggle "Notebook access" ON pra este notebook')
    print('  5. Re-rode esta célula')
    print('=' * 70)

if not REPO_ROOT.exists():
    try:
        subprocess.run(['git', 'clone', '--depth=1', '--branch', BRANCH,
                        clone_url, str(REPO_ROOT)], check=True)
    except subprocess.CalledProcessError:
        explain_auth_failure()
        raise
else:
    # Re-aponta o remote com auth caso o repo já existir de antes
    subprocess.run(['git', '-C', str(REPO_ROOT), 'remote', 'set-url',
                    'origin', clone_url], check=False)
    try:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'reset', '--hard',
                        f'origin/{BRANCH}'], check=True)
    except subprocess.CalledProcessError:
        explain_auth_failure()
        raise

os.environ['REPO_ROOT'] = str(REPO_ROOT)
os.chdir(REPO_ROOT)
subprocess.run(['git', 'log', '--oneline', '-1'], check=True)


In [ ]:
sys.path.insert(0, str(REPO_ROOT / 'colab'))

# 80m scale — usamos o scale label mas o ModelConfig abaixo decide o tamanho real
os.environ['NSOS_PARAM_SCALE'] = '80m'
# ATIVA BF16 Tensor Cores (essencial pra caber 200M em T4)
os.environ['NSOS_MIXED_PRECISION'] = 'bf16'

import colab_bootstrap
env = colab_bootstrap.setup_environment(verbose=True)
BUILD_DIR = Path(env['build_dir'])
EXT_SO = Path(env['ext_so'])
print(f'\nBuild OK on {env["gpu_name"]}')
print(f'BF16: {os.environ.get("NSOS_MIXED_PRECISION")}')

In [ ]:
sys.path.insert(0, str(BUILD_DIR))
if 'nsos_ext' in sys.modules:
    del sys.modules['nsos_ext']
import nsos_ext
print(f'nsos_ext from: {nsos_ext.__file__}')
print(f'has QATScheduler: {hasattr(nsos_ext, "QATScheduler")}')
print(f'has fast_gpu_supported: {nsos_ext.fast_gpu_supported() if hasattr(nsos_ext, "fast_gpu_supported") else "n/a"}')

## 2. Dataset SMALL — só o suficiente pra ver loss descer

Para smoke test não precisamos 3GB. 500MB de texto cobre 500-1000 steps facilmente.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'datasets>=2.16', 'huggingface_hub', 'tqdm'], check=True)

DATA_ROOT = Path('/content/drive/MyDrive/oxta_smoke_data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# ── HuggingFace auth: CulturaX exige rate-limit-friendly token
# Setup: igual ao GITHUB_TOKEN — Secrets do Colab, nome HF_TOKEN
# Pega em https://huggingface.co/settings/tokens (token READ é suficiente)
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
        print('[auth] HF_TOKEN configurado via Colab Secrets')
    else:
        print('[auth] sem HF_TOKEN — CulturaX pode falhar por rate limit (sem token)')
        print('       BR-TaxQA-R deve baixar normalmente (público sem auth)')
except Exception:
    print('[auth] userdata.get falhou — seguindo sem HF_TOKEN')

# Só CulturaX (500MB) + BR-TaxQA-R (185MB, já tem CARF embutido)
# Skip BACEN FAQ + LeNER-Br pra ser rápido
cmd = [
    sys.executable,
    str(REPO_ROOT / 'OXN/nsos/scripts/oxta_contabil/download_datasets.py'),
    '--output-dir', str(DATA_ROOT),
    '--include', 'culturax_ptbr,br_taxqa',
    '--target-bytes', '500MB',
    '--culturax-shards', '10',
]
print(f'\nRunning: {" ".join(cmd[-6:])}\n')
result = subprocess.run(cmd, check=False)
print(f'\nexit code: {result.returncode}')
print('NOTA: exit=2 significa "1+ dataset falhou" — se BR-TaxQA-R passou')
print('      o smoke test ainda roda com 7-8k docs do CARF (suficiente).')


In [ ]:
import json
from typing import Iterator

RAW_DIR = DATA_ROOT / 'oxta_contabil' / 'raw'
CORPUS_PATH = DATA_ROOT / 'corpus_smoke.jsonl'
MIN_TEXT_LEN = 200

def extract_culturax(root: Path) -> Iterator[str]:
    for shard in sorted(root.glob('culturax_ptbr_shard_*.jsonl')):
        with open(shard, encoding='utf-8') as f:
            for line in f:
                try:
                    row = json.loads(line)
                except json.JSONDecodeError:
                    continue
                text = (row.get('text') or '').strip()
                if len(text) >= MIN_TEXT_LEN:
                    yield text

def extract_br_taxqa(root: Path) -> Iterator[str]:
    # CARF é o tesouro — texto técnico-fiscal real, ~173MB
    for f in sorted(root.glob('acordaos_CARF_*.json')):
        data = json.loads(f.read_text(encoding='utf-8'))
        items = data if isinstance(data, list) else list(data.values())
        for item in items:
            if isinstance(item, dict):
                text = (item.get('text') or item.get('filedata') or '').strip()
                if len(text) >= MIN_TEXT_LEN:
                    yield text
    # Q&A + refs também
    for f in sorted(root.glob('questions_QA_*.json')):
        data = json.loads(f.read_text(encoding='utf-8'))
        items = data['questions'] if isinstance(data, dict) and 'questions' in data \
                else data if isinstance(data, list) else list(data.values())
        for item in items:
            if isinstance(item, dict):
                q = (item.get('question_text') or item.get('question') or '').strip()
                a = item.get('answer') or item.get('answer_cleaned') or ''
                if isinstance(a, list):
                    a = '\n'.join(str(x) for x in a)
                text = (q + '\n\n' + str(a).strip()).strip()
                if len(text) >= MIN_TEXT_LEN:
                    yield text

total_docs = 0
total_chars = 0
with open(CORPUS_PATH, 'w', encoding='utf-8') as out:
    for name, extractor in [('culturax_ptbr', extract_culturax),
                              ('br_taxqa', extract_br_taxqa)]:
        d = RAW_DIR / name
        if not d.exists():
            print(f'[skip] {name}')
            continue
        n, chars = 0, 0
        for text in extractor(d):
            out.write(json.dumps({'text': text}, ensure_ascii=False) + '\n')
            n += 1
            chars += len(text)
        total_docs += n
        total_chars += chars
        print(f'[ok] {name:20s} {n:>6d} docs, {chars/1024**2:>7.1f} MB')
print(f'\nTOTAL: {total_docs:,} docs, {total_chars/1024**2:.1f} MB texto')
print(f'corpus em: {CORPUS_PATH}')

## 3. ModelConfig — ~200M params com FULL STACK

Setup explícito de TODAS as techs de produção. Anti-pattern: não confiar nos defaults.

In [ ]:
config = nsos_ext.ModelConfig()

# ── Arquitetura target ~200M params
config.num_layers          = 12
config.d_model             = 768       # vs 320 do 40M
config.vocab_size          = 16384
config.max_context_tokens  = 1024

# ── Attention (GQA): 3 de 12 layers (slot 3 de period 4)
config.n_heads             = 12
config.n_kv_heads          = 3         # GQA 4:1
config.sliding_window      = 2048
config.attention_period    = 4
config.attention_slot      = 3
config.use_exact_attention_training = True
config.use_flash_attn      = False     # T4 não suporta

# ── MoE-8 top-2: 2 de 12 layers (slot 5 de period 6)
config.use_moe             = True
config.num_experts         = 8
config.num_experts_per_token = 2
config.moe_period          = 6
config.moe_slot            = 5

# ── Research-only: OFF
config.use_ttt             = False

# ── Memória
config.use_gradient_checkpointing = True
config.dropout             = 0.0
config.use_cuda            = True
config.default_batch_size  = 1         # smoke: batch=1 cabe folgado em 16GB

# ── MCTS é inference-only
config.mcts_simulations    = 0
config.mcts_depth          = 0

# Estimativa rough de params:
d = config.d_model
L = config.num_layers
V = config.vocab_size
n_attn = L // config.attention_period
n_moe = L // config.moe_period
n_mamba = L - n_attn - n_moe
embed = 2 * V * d
mamba_p = n_mamba * (4 * d * d + 2 * d * 128)
attn_p = n_attn * (2 * d * d + 2 * d * (d // 4))
moe_p = n_moe * config.num_experts * 2 * d * (4 * d)
total_p = embed + mamba_p + attn_p + moe_p

print(f'Layer composition:')
print(f'  Mamba2: {n_mamba} layers')
print(f'  Attn:   {n_attn} layers (period={config.attention_period}, slot={config.attention_slot})')
print(f'  MoE:    {n_moe} layers ({config.num_experts} experts, top-{config.num_experts_per_token})')
print(f'\nParam estimate (rough):')
print(f'  embed+head:  {embed/1e6:6.1f}M')
print(f'  mamba:       {mamba_p/1e6:6.1f}M')
print(f'  attention:   {attn_p/1e6:6.1f}M')
print(f'  moe experts: {moe_p/1e6:6.1f}M')
print(f'  TOTAL:       {total_p/1e6:6.1f}M (alvo ~200M)')

## 4. Engine + QAT progressivo (1.58-bit)

In [ ]:
engine = nsos_ext.InferenceEngine()

# load_model(path, config): path="" = fresh init com o config dado.
# Passando um nome de profile aqui (ex: "hybrid_pilot") faz o engine
# tentar abrir um ARQUIVO chamado "hybrid_pilot" — que não existe e
# retorna False. A arquitetura vem 100% do ModelConfig acima.
print(f'Inicializando engine fresh com config acima...')
ok = engine.load_model('', config)
print(f'load_model: {"OK" if ok else "FALHOU"}')

if ok:
    mem = engine.get_memory_usage()
    print(f'Memory após init: {mem}')
else:
    raise RuntimeError('Engine init falhou — checa ModelConfig acima (provavelmente um campo inválido)')

# QAT progressivo: warmup FP → ternary kicks in
# Nota: este binding (commit fc1de80) NÃO expõe QATScheduler nem engine.trainer
# via Python. QAT progressivo só é configurável via train_curriculum.py
# subprocess. Para smoke test isso é OK — modelo treina em FP/BF16
# e exercita Mamba2 + Attention + MoE + BF16.
qat_ok = False
if hasattr(nsos_ext, 'QATScheduler') and hasattr(engine, 'trainer'):
    try:
        qat = nsos_ext.QATScheduler()
        qat.enabled = True
        qat.semantic_warmup_steps = 80
        qat.qat_start_step = 160
        qat.quantized_precision_bits = 2
        qat.ternary_regularization = 1.5e-4
        engine.trainer.configure_progressive_qat(qat)
        print(f'\nQAT 1.58-bit: ON (warmup 80 → ternary 160)')
        qat_ok = True
    except Exception as e:
        print(f'\nQAT setup falhou: {e}')

if not qat_ok:
    print(f'\nQAT scheduler: NÃO exposto neste binding (treino vai correr em BF16)')
    print('  → para 1.58-bit, usar train_curriculum.py via subprocess depois')


## 5. Geração ANTES do treino (baseline gibberish esperado)

In [ ]:
BASELINE_PROMPTS = [
    'O Imposto de Renda',
    'A nota fiscal eletrônica',
    'O contribuinte deve',
]

# generate(prompt, max_tokens, temperature) — 3 args posicionais.
# Para usar GenerationOptions completo, é generate_ex(prompt, options).
# Aqui escolhemos o simples pra não acoplar com GenerationOptions schema.
MAX_TOKENS = 50
TEMPERATURE = 0.8

baseline = {}
for p in BASELINE_PROMPTS:
    try:
        baseline[p] = engine.generate(p, MAX_TOKENS, TEMPERATURE)
    except Exception as e:
        baseline[p] = f'(erro: {type(e).__name__}: {e})'
    print(f'[{p!r}]')
    print(f'  -> {baseline[p][:200]}\n')


## 6. Treino smoke — 500 steps com diagnósticos

Loop direto: `engine.train_text(doc)` em cada doc. Log detalhado a cada 25 steps.

In [ ]:
import time
import random
import matplotlib.pyplot as plt

MAX_STEPS = int(os.environ.get('NSOS_SMOKE_STEPS', '500'))
LOG_EVERY = 25

# Carrega corpus em memória
docs = []
with open(CORPUS_PATH, encoding='utf-8') as f:
    for line in f:
        try:
            docs.append(json.loads(line)['text'])
        except (json.JSONDecodeError, KeyError):
            continue
print(f'Docs no corpus: {len(docs):,}')

random.seed(1337)
random.shuffle(docs)

losses = []
step_times = []
memory_samples = []
t_start = time.time()

for step in range(1, MAX_STEPS + 1):
    t0 = time.time()
    doc = docs[(step - 1) % len(docs)]
    try:
        result = engine.train_text(doc[:3000])  # cap 3000 chars
        loss = float(result) if isinstance(result, (int, float)) else float(result.get('loss', 'nan'))
    except Exception as e:
        print(f'[step {step}] train_text falhou: {type(e).__name__}: {e}')
        continue
    step_times.append(time.time() - t0)
    losses.append(loss)

    if step % LOG_EVERY == 0 or step == 1:
        recent = losses[-LOG_EVERY:]
        avg_loss = sum(recent) / len(recent)
        avg_step = sum(step_times[-LOG_EVERY:]) / min(LOG_EVERY, len(step_times))
        eta_min = (MAX_STEPS - step) * avg_step / 60
        mem = engine.get_memory_usage()
        memory_samples.append((step, mem))
        qat_marker = ''
        if qat_ok:
            if step < 80:
                qat_marker = ' [QAT: FP warmup]'
            elif step < 160:
                qat_marker = ' [QAT: pre-ternary]'
            else:
                qat_marker = ' [QAT: TERNARY ACTIVE]'
        print(f'[step {step:>4d}/{MAX_STEPS}] loss={avg_loss:.3f}  '
              f'step={avg_step:.2f}s  eta={eta_min:.1f}min{qat_marker}')

total_min = (time.time() - t_start) / 60
print(f'\nFIM: {len(losses)} steps em {total_min:.1f} min')
if losses:
    print(f'  loss inicial (primeiros 10): {sum(losses[:10])/min(10,len(losses)):.3f}')
    print(f'  loss final   (últimos 25):   {sum(losses[-25:])/min(25,len(losses)):.3f}')

## 7. Verdict — passou ou falhou?

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(losses, alpha=0.4, label='step loss')
window = max(1, len(losses) // 25)
if len(losses) > window:
    smoothed = [sum(losses[max(0, i-window):i+1]) / min(i+1, window) for i in range(len(losses))]
    plt.plot(smoothed, color='red', linewidth=2, label=f'smoothed (w={window})')
if qat_ok:
    plt.axvline(80, color='orange', linestyle='--', alpha=0.5, label='QAT FP→pre-ternary')
    plt.axvline(160, color='red', linestyle='--', alpha=0.5, label='QAT ternary ACTIVE')
plt.xlabel('step')
plt.ylabel('loss')
plt.title(f'Oxta Stack Smoke — 200M params, full stack ({len(losses)} steps)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/oxta_smoke_data/stack_smoke_loss.png', dpi=110)
plt.show()

# Verdict numérico
import math
if not losses:
    verdict = 'FALHA: nenhum step completou'
else:
    loss_initial = sum(losses[:10]) / min(10, len(losses))
    loss_final = sum(losses[-25:]) / min(25, len(losses))
    drop = loss_initial - loss_final
    drop_pct = (drop / loss_initial * 100) if loss_initial > 0 else 0

    # Critérios:
    # - loss inicial deve ser ~ln(vocab) = ln(16384) ≈ 9.7 (sanity inicial)
    # - loss final deve ser < loss inicial (modelo aprendeu)
    # - drop > 15% indica aprendizado real, não ruído
    nan_count = sum(1 for l in losses if math.isnan(l) or math.isinf(l))
    expected_initial = math.log(config.vocab_size)

    if nan_count > len(losses) * 0.05:
        verdict = f'❌ FALHA: {nan_count} losses NaN/Inf (>5% do treino)'
    elif loss_initial > expected_initial * 2 or loss_initial < 0.1:
        verdict = (f'⚠️ ANORMAL: loss inicial {loss_initial:.2f} fora do range '
                    f'esperado (~{expected_initial:.1f})')
    elif drop_pct < 15:
        verdict = (f'⚠️ INSUFICIENTE: loss caiu só {drop_pct:.1f}% '
                    f'({loss_initial:.2f} → {loss_final:.2f}). '
                    f'Pode precisar mais steps ou LR maior.')
    else:
        verdict = (f'✅ PASSOU: loss caiu {drop_pct:.1f}% '
                    f'({loss_initial:.2f} → {loss_final:.2f}). '
                    f'Stack funcional, modelo aprendeu.')

print(f'\n{"="*70}')
print(f'VERDICT: {verdict}')
print(f'{"="*70}')
print(f'\nMetrics:')
if losses:
    print(f'  steps completados:  {len(losses)}/{MAX_STEPS}')
    print(f'  loss inicial (10):  {sum(losses[:10])/min(10,len(losses)):.3f}')
    print(f'  loss final (25):    {sum(losses[-25:])/min(25,len(losses)):.3f}')
    print(f'  loss min:           {min(losses):.3f}')
    print(f'  loss max:           {max(losses):.3f}')
    print(f'  losses NaN/Inf:     {nan_count}')
    print(f'  vocab esperado ln:  {expected_initial:.2f}')
    print(f'  tempo total:        {total_min:.1f}min')
    print(f'  step médio:         {sum(step_times)/len(step_times):.2f}s')

## 8. Geração DEPOIS — vê se algo mudou

200M aprendendo de 500 steps em PT-BR fiscal ainda vai gerar coisa ruim — esperado. O que queremos ver: **mudou de aleatório pra ter alguma cara de PT-BR**.

In [ ]:
for p in BASELINE_PROMPTS:
    print(f'\n{"=" * 70}')
    print(f'PROMPT: {p!r}')
    print(f'\n[ANTES] {baseline[p][:200]}')
    try:
        after = engine.generate(p, MAX_TOKENS, TEMPERATURE)
    except Exception as e:
        after = f'(erro: {type(e).__name__}: {e})'
    print(f'\n[DEPOIS] {after[:200]}')


## 9. Salvar pack (smoke checkpoint)

Se passou no verdict, salva o pack pra reusar em sessões futuras.

In [ ]:
from datetime import datetime

PACK_DIR = Path('/content/drive/MyDrive/oxta_smoke_packs') / \
           f'smoke_200m_{datetime.now().strftime("%Y%m%d_%H%M")}'
PACK_DIR.mkdir(parents=True, exist_ok=True)

try:
    engine.save_model_pack(str(PACK_DIR))
    print(f'Pack salvo: {PACK_DIR}')
    total_mb = sum(f.stat().st_size for f in PACK_DIR.rglob('*') if f.is_file()) / 1024**2
    print(f'Tamanho:    {total_mb:.1f} MB')
except Exception as e:
    print(f'save_model_pack falhou: {e}')
    try:
        engine.save_checkpoint(str(PACK_DIR / 'checkpoint.nsos'))
        print(f'Fallback OK: checkpoint salvo')
    except Exception as e2:
        print(f'Fallback também falhou: {e2}')

## 10. Próximos passos

### Se passou (✅)
- Stack confirmado funcional em 200M
- Pode partir pro pre-training de produção: `oxta_contabil_pretrain.ipynb` com `NSOS_MAX_STEPS=50000`
- Ou escalar pra 400M-1B aumentando `d_model=1024` e `num_layers=16`

### Se ficou insuficiente (⚠️)
Possíveis ajustes (mexer um por vez pra isolar causa):
- Aumentar `MAX_STEPS=1000` (talvez 500 foi pouco)
- Aumentar LR no trainer: `engine.trainer.learning_rate = 0.001` antes do loop
- Reduzir scale temporariamente: `d_model=512, num_layers=8` (~80M, mais fácil de aprender)
- Desligar MoE temporariamente: `config.use_moe = False` (isola se MoE quebrou)

### Se falhou (❌)
Manda o erro:
- NaN/Inf → provavelmente BF16 unstable, tentar `NSOS_MIXED_PRECISION=fp32`
- OOM → reduzir `d_model=640` ou `num_layers=10`
- Crash CUDA → kernel issue, mandar stack trace
- `load_model` retornando False → arquitetura inválida, voltar pra mamba_small e ir adicionando 1 tech por vez